웹페이지들의 링크 정보(A->B, A->C)가 주어졌을 때, 각 페이지의 영향력(Rank) 계산

핵심로직
1. 각 페이지는 자신의 현재 랭크를 연결된 이웃들에게세 1/N씩 나눠 줌(map 단계)
2. 각 페이지는 이웃들에게 받은 랭크를 모두 더함(reduce 단계)
3. 공식(0.15+0.85*sum)을 적용해 랭크를 갱신

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
  .master("local[*]")\
  .getOrCreate()
sc = spark.sparkContext

In [2]:
#1. 데이터 준비
links_list = [
    ("A", ["B", "C"]),
    ("B", ["C"]),
    ("C", ["A"]),
    ("D", ["C"])
]
#RDD 생성 links: (페이지 id, [연결된 페이지 목록])
links = sc.parallelize(links_list).persist()
#초기 랭크 설정->우선 모드 다 1.0으로 시작
#ranks: (페이지 id, 현재 랭크값)
ranks = links.map(lambda x: (x[0], 1.0))

In [3]:
#핵심 map 함수 urls:연결된 애들 rank: 지금 내 점수
def compute(urls, rank):
  num_urls = len(urls)
  #내 점수를 연결된 개수만큼 나눠서 보냄
  for url in urls:
    yield(url, rank / num_urls)

In [4]:
#PageRank알고리즘 10회 반복
for i in range(10):
  #join: 랭크 정보와 현재 랭크를 합침
  ranks1 = links.join(ranks).flatMap(
      lambda x: compute(x[1][0], x[1][1])
  )
  #reduce: 받은 점수 합산
  #결과:(페이지 id, 받은 점수 합계)
  ranks2 = ranks1.reduceByKey(lambda a, b: a+b).mapValues(lambda rank: 0.15+0.85*rank)

In [6]:
print(ranks2.collect())

[('B', 0.575), ('A', 1.0), ('C', 2.275)]
